In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('data/mammographic_mass.csv')

# lowercase column names
df.columns = df.columns.str.lower().str.replace('-', '_')

# bi_rads is the radiologist's own rating (non-predictive in UCI docs),
# so we drop it to avoid data leakage
df = df.drop(columns=['bi_rads'])
df.head()

,age,shape,margin,density,severity
0,67.0,3.0,5.0,3.0,1
1,43.0,1.0,1.0,NaN,1
2,58.0,4.0,5.0,3.0,1
3,28.0,1.0,1.0,3.0,0
4,74.0,1.0,5.0,NaN,1


In [4]:
df_full_train,df_test=train_test_split(df,test_size=0.2,random_state=1)
df_train,df_val=train_test_split(df_full_train,test_size=0.25,random_state=1)

df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

y_train=df_train.severity.values
y_val=df_val.severity.values
y_test=df_test.severity.values

del df_train['severity']
del df_val['severity']
del df_test['severity']



In [5]:
print(len(df_train), len(df_val), len(df_test))
print(df_train.columns.tolist())

576 192 193
['age', 'shape', 'margin', 'density']


In [6]:
df_full_train = df_full_train.reset_index(drop=True)

print(df_full_train.isna().sum())
print()
print(df_full_train.describe())

age          3
shape       26
margin      34
density     59
severity     0
dtype: int64

              age       shape      margin     density    severity
count  765.000000  742.000000  734.000000  709.000000  768.000000
mean    55.707190    2.739892    2.844687    2.919605    0.471354
std     14.302107    1.245125    1.580187    0.372837    0.499504
min     19.000000    1.000000    1.000000    1.000000    0.000000
25%     45.000000    2.000000    1.000000    3.000000    0.000000
50%     57.000000    3.000000    3.000000    3.000000    0.000000
75%     66.000000    4.000000    4.000000    3.000000    1.000000
max     96.000000    4.000000    5.000000    4.000000    1.000000


In [7]:
print(df_full_train.severity.value_counts())
print(df_full_train.severity.value_counts(normalize=True))

severity
0    406
1    362
Name: count, dtype: int64
severity
0    0.528646
1    0.471354
Name: proportion, dtype: float64


In [8]:
for col in ['shape', 'margin', 'density']:
    print(col)
    print(df_full_train.groupby(col).severity.agg(['mean', 'count']))
    print()

shape
           mean  count
shape                 
1.0    0.179775    178
2.0    0.179012    162
3.0    0.428571     77
4.0    0.790769    325

margin
            mean  count
margin                 
1.0     0.125000    280
2.0     0.631579     19
3.0     0.608696     92
4.0     0.674208    221
5.0     0.844262    122

density
             mean  count
density                 
1.0      0.454545     11
2.0      0.319149     47
3.0      0.500782    639
4.0      0.416667     12



In [9]:
from sklearn.metrics import mutual_info_score

for col in ['shape', 'margin', 'density']:
    score = mutual_info_score(df_full_train[col].fillna(-1), df_full_train.severity)
    print(col, round(score, 3))
print(df_full_train.groupby('severity').age.describe())    

shape 0.175
margin 0.185
density 0.01
          count       mean        std   min    25%   50%   75%   max
severity                                                            
0         406.0  49.519704  13.274439  19.0  40.25  51.0  59.0  86.0
1         359.0  62.704735  12.027518  31.0  55.00  64.0  71.0  96.0


In [10]:
print(df_full_train.severity.value_counts())

severity
0    406
1    362
Name: count, dtype: int64


In [11]:
categorical = ['shape', 'margin', 'density']
numerical = ['age']

age_median = df_train.age.median()

def prepare(df):
    df = df.copy()
    df['age'] = df['age'].fillna(age_median)
    for col in categorical:
        # missing -> 'unknown'
        df[col] = df[col].apply(lambda v: 'unknown' if pd.isna(v) else str(int(v)))
    return df


df_train = prepare(df_train)
df_val = prepare(df_val)
df_test = prepare(df_test)

print(age_median)
print(df_train.isna().sum())
print(df_train.head())

57.0
age        0
shape      0
margin     0
density    0
dtype: int64
    age shape margin  density
0  77.0     4      5        3
1  50.0     3      2        3
2  63.0     2      1  unknown
3  35.0     1      1        2
4  56.0     4      4        3


In [12]:
from sklearn.feature_extraction import DictVectorizer
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')
dv=DictVectorizer(sparse=False)
X_train=dv.fit_transform(train_dicts)
X_val=dv.transform(val_dicts)
print(X_train.shape, X_val.shape)
print(dv.get_feature_names_out())

(576, 17) (192, 17)
['age' 'density=1' 'density=2' 'density=3' 'density=4' 'density=unknown'
 'margin=1' 'margin=2' 'margin=3' 'margin=4' 'margin=5' 'margin=unknown'
 'shape=1' 'shape=2' 'shape=3' 'shape=4' 'shape=unknown']


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)
y_pred=model.predict_proba(X_val)[:,1]
roc_auc_score(y_val,y_pred)

0.8797653958944281